# Case study — Building heating energy regression

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import statsmodels.formula.api as smf
from scipy import stats

RANDOM_STATE = 42
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

## 1. Data loading and global audit

In [ ]:
df_raw = pd.read_csv("building_energy_regression_case.csv")

display(df_raw.head())
print("Shape:", df_raw.shape)

In [ ]:
display(df_raw.select_dtypes(include=np.number).describe().T)

In [ ]:
display(df_raw.select_dtypes(include=["object", "category", "bool"]).describe().T)

Few observations:
- Target is `heating_energy_kwh_per_sqm` and is directly computable from `total_annual_heating_kwh` — we need to remove the latter to prevent leakage 
- 1,200 observations, relatively few vs. size of the feature space: 11 numerical features and 10 categorical features each with 2 to 5 categories — dimensionality will be an issue and regularisation / CV will be key to prevent overfitting
- Min-max ranges are plausible for all numerical features — no reason to aggressively remove or winsorize potential outliers
- No duplicated rows, each building ID is unique — no need to remove duplicates

## 2. Leakage handling and protected holdout

In [ ]:
df = df_raw.drop_duplicates().copy()
df = df.drop('total_annual_heating_kwh', axis=1)

# ── Convert categoricals to string (avoids bool/str mix issues in SimpleImputer) ──
cat_cols_raw = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in cat_cols_raw:
    df[col] = df[col].astype(str)

print("Raw shape:", df_raw.shape)
print("Post-dedup shape:", df.shape)
print("Duplicates removed:", len(df_raw) - len(df))

In [ ]:
dev_df, holdout_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("Dev:    ", dev_df.shape)
print("Holdout:", holdout_df.shape)

**240 apartments** set aside as a protected holdout (20%). The remaining **960 rows** are used for all EDA, feature engineering, and model selection.

## 3. EDA — dev sample only

### 3.1 Target distribution 

In [ ]:
target_summary = dev_df['heating_energy_kwh_per_sqm'].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).to_frame()
display(target_summary)

In [ ]:
dev_df[dev_df['heating_energy_kwh_per_sqm'] == 310]['building_id'].count()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(dev_df["heating_energy_kwh_per_sqm"], bins=50, edgecolor="white", linewidth=0.3)
axes[0].set_title("heating_energy_kwh_per_sqm distribution")
axes[0].set_xlabel("kwh / sqm")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))

axes[1].scatter(dev_df["floor_area_sqm"], dev_df["heating_energy_kwh_per_sqm"], alpha=0.2, s=12)
axes[1].set_title("Surface vs unit consumption")
axes[1].set_xlabel("Surface (sqm)")
axes[1].set_ylabel("kwh / sqm")

plt.tight_layout()
plt.show()

Nothing specific to note: target distribution looks well enough balanced and we can guess a slight negative correlation between surface and target. As we will keep surface in the feature mix, our models will be able to factor this in.

To be noted: target looks to have been winsorized as we have 2% of our rows exactly at 310 kwh/m2 which seems very unlikely. Given it is only 2% this should have no material impact on our model.

### 3.2 Missing values — volume and cons impact

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": dev_df.isna().sum(),
    "missing_pct": dev_df.isna().mean().mul(100).round(2),
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

display(missing_summary)

missing_cols = missing_summary.index.tolist()


In [ ]:
# ── Core question: do apartments with missing data have different kwh/m2? ──
kwh_impact_rows = []
for col in missing_cols:
    mask = dev_df[col].isna()
    kwh_missing = dev_df.loc[mask, "heating_energy_kwh_per_sqm"]
    kwh_present = dev_df.loc[~mask, "heating_energy_kwh_per_sqm"]
    
    # Welch t-test (unequal variances)
    t_stat, p_val = stats.ttest_ind(kwh_missing, kwh_present, equal_var=False)
    
    kwh_impact_rows.append({
        "feature": col,
        "n_missing": mask.sum(),
        "mean_kwh_missing": kwh_missing.mean(),
        "mean_kwh_present": kwh_present.mean(),
        "delta_kwh": kwh_missing.mean() - kwh_present.mean(),
        "delta_pct": (kwh_missing.mean() - kwh_present.mean()) / kwh_present.mean() * 100,
        "t_stat": t_stat,
        "p_value": p_val,
    })

kwh_impact = pd.DataFrame(kwh_impact_rows).sort_values("delta_kwh", key=abs, ascending=False)
for c in ["mean_kwh_missing", "mean_kwh_present", "delta_kwh"]:
    kwh_impact[c] = kwh_impact[c].round(0).astype(int)
kwh_impact["delta_pct"] = kwh_impact["delta_pct"].round(1)
kwh_impact["p_value"] = kwh_impact["p_value"].map(lambda x: f"{x:.4f}")

display(kwh_impact)

Missingness on insulation and airtightness seem to be informative on kwh/sqm, which would make sense given that recently diagnosed apartments could tend to perform better on those dimensions. The question is whether this information subsists when we look at a more granular level based on the other features.

Given the time constraints and the dimensionality issue we will assume it does not survive a more granular test so we don't need to add missingness features during imputation.

### 3.3 Kwhpsqm gradients by categorical segment

In [ ]:
categorical_cols_for_eda = dev_df.select_dtypes(include=["object", "category", "bool"]).columns[1:]

for col in categorical_cols_for_eda:
    segment = (
        dev_df.groupby(col, dropna=False)
        .agg(
            count=("heating_energy_kwh_per_sqm", "size"),
            median_kwhpsqm=("heating_energy_kwh_per_sqm", "median"),
            mean_kwhpsqm=("heating_energy_kwh_per_sqm", "mean"),
            std_kwhpsqm=("heating_energy_kwh_per_sqm", "std"),
            median_surface=("floor_area_sqm", "median"),
        )
        .sort_values("median_kwhpsqm", ascending=False)
    )
    print(f"\n── {col} ──")
    display(segment.round())

Link between categorical features and target makes sense. The most informative categorical features are:
- `insulation_grade`
- `construction_period`

We can note that `solar_shading` has a negative impact on kwh/sqm which can seem odd as shading should reduce solar heat thus increase the need for artificial heating. However we can imagine that apartments equipped with solar shading are on average more recent and better insulated — to be confirmed once we get regression coefficients on the whole dataset.

### 3.4 Numerical correlations and nonlinearities

In [ ]:
numeric_cols = dev_df.select_dtypes(include=np.number).columns

corr = dev_df[numeric_cols].corr(numeric_only=True)["heating_energy_kwh_per_sqm"].drop("heating_energy_kwh_per_sqm").sort_values()
display(corr.to_frame("corr_with_kwhpsqm"))

In [ ]:
top_col = abs(corr).sort_values().tail(6).index.to_list()
top_col

In [ ]:
# ── Check for nonlinearity in the top numeric features ──
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for ax, col in zip(axes, top_col):
    ax.scatter(dev_df[col], dev_df["heating_energy_kwh_per_sqm"], alpha=0.15, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel("heating_energy_kwh_per_sqm")
    ax.set_title(f"r = {dev_df[[col, 'heating_energy_kwh_per_sqm']].corr().iloc[0,1]:.3f}")

plt.tight_layout()
plt.show()

In [ ]:
# checking colinearity 
import seaborn as sns

sns.heatmap(dev_df[numeric_cols].drop("heating_energy_kwh_per_sqm", axis=1).corr(numeric_only=True))
dev_df[numeric_cols].drop("heating_energy_kwh_per_sqm", axis=1).corr(numeric_only=True)

Overall we see strong signal in several features and low risk of collinearity, except between `rooms_count` and `floor_area_sqm` which are not in the top 6 signal.

**EDA takeaway → feature selection:** given the dimensionality issue (1,200 obs, 21 features) and the high concentration of the signal in 6 numerical features and 2 categorical features, we will train models on this reduced feature set to improve generalisation. We also add one engineered feature (`log_floor_area_sqm`) motivated by the concave surface-vs-target relationship observed in §3.1.

## 4. Feature engineering, feature selection, and train/test split

In [ ]:
# ── Feature engineering (applied to full dev_df before split) ──
dev_df["log_floor_area_sqm"] = np.log1p(dev_df["floor_area_sqm"])
holdout_df["log_floor_area_sqm"] = np.log1p(holdout_df["floor_area_sqm"])

# ── Feature selection: top 6 numeric + log_floor_area + top 2 categorical ──
top_num = abs(corr).sort_values().tail(6).index.tolist()
selected_numeric = top_num + ["log_floor_area_sqm"]
selected_categorical = ["insulation_grade", "construction_period"]
selected_features = selected_numeric + selected_categorical

print(f"Selected features ({len(selected_features)}):")
print(f"  Numeric:     {selected_numeric}")
print(f"  Categorical: {selected_categorical}")

# ── Train/test split (80/20 of dev) ──
train_df, test_df = train_test_split(
    dev_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"\nTrain:  {train_df.shape}")
print(f"Test:   {test_df.shape}")
print(f"Holdout:{holdout_df.shape}")

In [ ]:
train_df.describe(include='all').T

In [ ]:
target = "heating_energy_kwh_per_sqm"

excluded_cols = ["building_id", "heating_energy_kwh_per_sqm"]

# ── All features ──
all_features = [col for col in train_df.columns if col not in excluded_cols]

# ── Selected features (top 6 numeric + log_floor_area + top 2 categorical) ──
top_num = abs(corr).sort_values().tail(6).index.tolist()
selected_features = top_num + ["log_floor_area_sqm"] + ["insulation_grade", "construction_period"]

print(f"All features:      {len(all_features)}")
print(f"Selected features: {len(selected_features)} -> {selected_features}")

# ── Default X/y on all features (for preprocessing definition) ──
X_train = train_df[all_features]
y_train = train_df[target]

X_test = test_df[all_features]
y_test = test_df[target]

X_holdout = holdout_df[all_features]
y_holdout = holdout_df[target]

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# ── Selected subsets ──
numeric_features_sel = [f for f in selected_features if f in numeric_features]
categorical_features_sel = [f for f in selected_features if f in categorical_features]

print(f"\nAll — {len(numeric_features)} numeric, {len(categorical_features)} categorical")
print(f"Selected — {len(numeric_features_sel)} numeric, {len(categorical_features_sel)} categorical")

## 5. Preprocessing and candidate models

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# ── Preprocessor for all features ──
preprocessor_all = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# ── Preprocessor for selected features ──
preprocessor_sel = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features_sel),
        ("cat", categorical_transformer, categorical_features_sel),
    ]
)

**Candidate models** cross two model families (Ridge, Random Forest) with two feature sets (all features, selected top features) to test whether (a) nonlinear interactions improve over main effects, and (b) feature selection improves generalisation on this small dataset.

In [ ]:
model_configs = {
    "ridge_all": {
        "model": Ridge(),
        "param_grid": {"model__alpha": [1.0, 3.0, 10.0, 30.0]},
        "features": all_features,
        "preprocessor": preprocessor_all,
    },
    "ridge_top": {
        "model": Ridge(),
        "param_grid": {"model__alpha": [1.0, 3.0, 10.0, 30.0]},
        "features": selected_features,
        "preprocessor": preprocessor_sel,
    },
    "rf_all": {
        "model": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
        "param_grid": {
            "model__max_depth": [None, 12, 20],
            "model__min_samples_leaf": [3, 8, 15],
            "model__max_features": [0.6, 0.8],
        },
        "features": all_features,
        "preprocessor": preprocessor_all,
    },
    "rf_top": {
        "model": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
        "param_grid": {
            "model__max_depth": [None, 12, 20],
            "model__min_samples_leaf": [3, 8, 15],
            "model__max_features": [0.6, 0.8],
        },
        "features": selected_features,
        "preprocessor": preprocessor_sel,
    },
}

## 6. Candidate model training and test-set evaluation
### 6.1 GridSearchCV on train, evaluate best on test

In [ ]:
best_estimators = {}
best_feature_sets = {}
results_rows = []

for name, cfg in model_configs.items():
    print(f"Fitting {name}...")
    feat = cfg["features"]
    pipe = Pipeline(steps=[
        ("preprocessor", cfg["preprocessor"]),
        ("model", cfg["model"]),
    ])
    gs = GridSearchCV(
        pipe,
        param_grid=cfg["param_grid"],
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    )
    gs.fit(train_df[feat], y_train)

    best = gs.best_estimator_
    best_estimators[name] = best
    best_feature_sets[name] = feat
    best_idx = gs.best_index_

    pred_train = best.predict(train_df[feat])
    pred_test = best.predict(test_df[feat])

    results_rows.append({
        "model": name,
        "n_features": len(feat),
        "best_params": gs.best_params_,
        "train_mae": round(mean_absolute_error(y_train, pred_train), 1),
        "cv_mae": round(-gs.cv_results_["mean_test_score"][best_idx], 1),
        "cv_mae_std": round(gs.cv_results_["std_test_score"][best_idx], 1),
        "test_mae": round(mean_absolute_error(y_test, pred_test), 1),
        "train_r2": round(r2_score(y_train, pred_train), 4),
        "test_r2": round(r2_score(y_test, pred_test), 4),
    })

model_results = pd.DataFrame(results_rows).sort_values("test_mae").reset_index(drop=True)
target_std_test = y_test.std()
model_results["mae_over_std"] = (model_results["test_mae"] / target_std_test).round(4)

display(model_results)
print(f"\nTest target std: {target_std_test:.0f} kwh/sqm")

**Ridge with all features wins** (MAE = 16.7 kwh/sqm, R² = 0.87). Two takeaways:

1. **Linear > tree-based.** RF overfits: train MAE 5–7 kwh/sqm vs test 21 kwh/sqm (3–4× gap). With only 768 training observations and 22 features, RF memorises the train set. Ridge's implicit regularisation is better suited to this low-n regime.

2. **Feature selection hurts.** Ridge on 9 features (MAE = 20.4) underperforms Ridge on all 22 features (MAE = 16.7). The 13 "minor" features collectively carry ~22% of additional signal. In a larger dataset, feature selection would likely help more — here, the penalty from dropping weak-but-real signals outweighs the variance reduction.

### 6.2 Naive benchmark and empirical prediction band

In [ ]:
best_row = model_results.iloc[0]
best_model_name = best_row["model"]
best_model_on_train = best_estimators[best_model_name]
best_feat = best_feature_sets[best_model_name]

best_test_pred = best_model_on_train.predict(test_df[best_feat])
best_test_abs_errors = np.abs(y_test.values - best_test_pred)

naive_pred = np.full(len(y_test), y_train.mean())
naive_mae = mean_absolute_error(y_test, naive_pred)
naive_rmse = mean_squared_error(y_test, naive_pred) ** 0.5

print(f"Naive baseline (train mean): MAE = {naive_mae:.0f} kwh/sqm, RMSE = {naive_rmse:.0f} kwh/sqm")
print(f"Selected model ({best_model_name}): MAE = {best_row['test_mae']:.0f} kwh/sqm")
print(f"Relative improvement: {1 - best_row['test_mae'] / naive_mae:.0%}")
print(f"\n95th-pct abs error: {np.percentile(best_test_abs_errors, 95):.0f} kwh/sqm (model)")
print(f"95th-pct abs error: {np.percentile(np.abs(y_test.values - naive_pred), 95):.0f} kwh/sqm (naive)")

The naive baseline (predict train mean for every apartment) yields **MAE ≈ 40 kwh/sqm**. The selected model reduces this by **~60%** — a very significant improvement.

The 95th-percentile absolute error gives a rough individual-prediction "uncertainty band": **±39 kwh/sqm** for the model vs ±103 kwh/sqm for the naive predictor. This is *not* a formal confidence interval — it is an empirical error quantile. For a proper uncertainty estimate, see the bootstrap analysis later.

## 7. Pre-holdout model interpretation

### 7.1 Ridge coefficient extraction

In [ ]:
# Use the best Ridge model for coefficient interpretation
ridge_name = "ridge_all" if "ridge_all" in best_estimators else "ridge_top"
for name in ["ridge_all", "ridge_top"]:
    if name in best_estimators:
        ridge_name = name  # take last (or pick best)

ridge_estimator = best_estimators[ridge_name]
ridge_feature_names = ridge_estimator.named_steps["preprocessor"].get_feature_names_out()
ridge_coefs = ridge_estimator.named_steps["model"].coef_

ridge_coef_table = (
    pd.DataFrame({
        "feature": ridge_feature_names,
        "coef_kwhpsqm": ridge_coefs,
        "abs_coef": np.abs(ridge_coefs),
    })
    .sort_values("abs_coef", ascending=False)
)

print(f"Ridge coefficients ({ridge_name}):")
display(ridge_coef_table.sort_values("coef_kwhpsqm", ascending=False))

### 7.2 OLS with HC3 robust standard errors

In [ ]:
ols_df = dev_df.copy()

# Fill missing for OLS (statsmodels doesn't handle NaN)
for col in categorical_features:
    ols_df[col] = ols_df[col].fillna("Missing")
for col in numeric_features:
    ols_df[col] = ols_df[col].fillna(ols_df[col].median())

formula = "heating_energy_kwh_per_sqm ~ " + " + ".join(numeric_features + [f"C({c})" for c in categorical_features])
ols_model = smf.ols(formula=formula, data=ols_df).fit(cov_type="HC3")

alpha = 0.05
coef_table = pd.DataFrame({
    "coef": ols_model.params,
    "std_err": ols_model.bse,
    "t_stat": ols_model.tvalues,
    "p_value": ols_model.pvalues,
})
coef_table["significant"] = coef_table["p_value"] < alpha
display(coef_table.sort_values("p_value"))

print(f"\nSignificant at {alpha}: {coef_table['significant'].sum()} / {len(coef_table)} coefficients")
print(f"R-squared: {ols_model.rsquared:.4f}")

HC3 OLS on the full dev set (R² = 0.876, consistent with Ridge). Key observations:

- **Dominant drivers:** `num_exterior_walls` (t = 24.5), `insulation_grade_very_poor` (t = 10.1), `insulation_thickness_cm` (t = -6.5) and `building_age_years` (t = 6.2). All directionally coherent with EDA.
- **`solar_shading` coefficient is negative** (less heating needed), confirming the EDA suspicion: it proxies building modernity, not a direct shading effect. Significant at 5%.
- **31 / 44 coefficients significant** — expected with 10 categorical features OHE-expanded. The non-significant coefficients are mostly low-prevalence category levels and weak numerical features (`window_to_wall_ratio`, `avg_indoor_temp_c`).

## 8. Final holdout evaluation
### 8.1 Refit on full dev set, evaluate on holdout

In [ ]:
# ── Refit best model on full dev set (train + test) before holdout evaluation ──
best_feat = best_feature_sets[best_model_name]
best_preproc = model_configs[best_model_name]["preprocessor"]

final_model = Pipeline(steps=[
    ("preprocessor", best_preproc),
    ("model", type(best_model_on_train.named_steps["model"])(**best_model_on_train.named_steps["model"].get_params())),
])
final_model.fit(dev_df[best_feat], dev_df[target])

holdout_pred_kwhpsqm = final_model.predict(holdout_df[best_feat])

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "holdout_mae_kwhpsqm": mean_absolute_error(y_holdout, holdout_pred_kwhpsqm),
    "holdout_rmse_kwhpsqm": mean_squared_error(y_holdout, holdout_pred_kwhpsqm) ** 0.5,
    "holdout_r2_kwhpsqm": r2_score(y_holdout, holdout_pred_kwhpsqm),
    "target_std_holdout": y_holdout.std(),
}])

holdout_metrics["mae_over_std"] = holdout_metrics["holdout_mae_kwhpsqm"] / holdout_metrics["target_std_holdout"]

display(holdout_metrics)

print(f"\nTest MAE:    {best_row['test_mae']:.0f} kwh/sqm")
print(f"Holdout MAE: {holdout_metrics['holdout_mae_kwhpsqm'].iloc[0]:.0f} kwh/sqm")
print(f"Degradation: {(holdout_metrics['holdout_mae_kwhpsqm'].iloc[0] - best_row['test_mae']) / best_row['test_mae']:.1%}")

In [ ]:
holdout_diag = pd.DataFrame({
    "actual_kwhpsqm": y_holdout.values,
    "predicted_kwhpsqm": holdout_pred_kwhpsqm,
})

holdout_diag["residual_kwhpsqm"] = holdout_diag["actual_kwhpsqm"] - holdout_diag["predicted_kwhpsqm"]
holdout_diag["abs_error_kwhpsqm"] = holdout_diag["residual_kwhpsqm"].abs()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Predicted vs actual
ax = axes[0]
ax.scatter(holdout_diag["actual_kwhpsqm"], holdout_diag["predicted_kwhpsqm"], alpha=0.3, s=15)
lims = [holdout_diag["actual_kwhpsqm"].min(), holdout_diag["actual_kwhpsqm"].max()]
ax.plot(lims, lims, "--", color="grey", linewidth=1)
ax.set_title("Predicted vs actual (kwhpsqm)")
ax.set_xlabel("Actual kwh/sqm")
ax.set_ylabel("Predicted kwh/sqm")

# Residual vs predicted (heteroscedasticity check)
ax = axes[1]
ax.scatter(holdout_diag["predicted_kwhpsqm"], holdout_diag["residual_kwhpsqm"], alpha=0.3, s=15)
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("Residual vs predicted")
ax.set_xlabel("Predicted kwh/sqm")
ax.set_ylabel("Residual kwh/sqm")

# Residual distribution
ax = axes[2]
ax.hist(holdout_diag["residual_kwhpsqm"], bins=50, edgecolor="white", linewidth=0.3)
ax.axvline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("Residual distribution")
ax.set_xlabel("Residual kwh/sqm")

plt.tight_layout()
plt.show()

Predicted-vs-actual hugs the diagonal with no systematic bias. The residual-vs-predicted plot shows no fan shape — **no evidence of heteroscedasticity**, unlike the apartment price model where the high-price segment was 3× noisier. The residual distribution is approximately symmetric and centered on zero, with light tails.

### 8.2 Bootstrap confidence intervals on holdout metrics

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
B = 2_000
n = len(y_holdout)

actual_arr = holdout_diag["actual_kwhpsqm"].to_numpy()
pred_arr = holdout_diag["predicted_kwhpsqm"].to_numpy()

boot_rows = []
for _ in range(B):
    s = rng.choice(n, size=n, replace=True)
    boot_rows.append({
        "mae_kwhpsqm": mean_absolute_error(actual_arr[s], pred_arr[s]),
        "rmse_kwhpsqm": mean_squared_error(actual_arr[s], pred_arr[s]) ** 0.5,
        "r2_kwhpsqm": r2_score(actual_arr[s], pred_arr[s]),
    })

boot_df = pd.DataFrame(boot_rows)
boot_summary = boot_df.quantile([0.025, 0.5, 0.975]).T
boot_summary.columns = ["ci_2.5%", "median", "ci_97.5%"]
display(boot_summary)

Bootstrap 95% CI (2,000 resamples): **MAE [14.1, 16.8] kwh/sqm**, **R² [0.846, 0.901]**. Intervals are tight relative to the point estimates — the holdout (n = 240) is large enough for precise metric estimation. The MAE CI excludes the naive baseline (45 kwh/sqm) by a wide margin.

### 8.3 Permutation test — is the model significantly better than chance?

In [ ]:
n_perm = 500
null_maes = []
actual_mae = mean_absolute_error(actual_arr, pred_arr)

for _ in range(n_perm):
    shuffled = rng.permutation(actual_arr)
    null_maes.append(mean_absolute_error(shuffled, pred_arr))

null_maes = np.array(null_maes)
p_value = (null_maes <= actual_mae).mean()

print(f"Actual holdout MAE:  {actual_mae:.1f} kwh/sqm")
print(f"Null MAE (mean):     {null_maes.mean():.1f} kwh/sqm")
print(f"Null MAE (min):      {null_maes.min():.1f} kwh/sqm")
print(f"Permutation p-value: {p_value:.4f}" + (f" (< {1/n_perm:.4f})" if p_value == 0 else ""))

Model MAE (15.5) is lower than all 500 null permutations (null mean = 59.9, null min = 53.3) — **p < 0.002**. Unsurprising given the gap, but it closes the statistical hygiene loop.

## 9. Final remarks

### Summary

The final model is a **Ridge regression** (alpha = 1.0) trained on all 22 features, refitted on the full development set (960 obs). It achieves **MAE = 15.5 kwh/sqm (R² = 0.877)** on the protected holdout — a 63% reduction over the naive baseline. Bootstrap 95% CI for MAE: [14.1, 16.8]. Permutation test: p < 0.002.

### Key insight: linear wins on small data

The 2×2 model comparison (Ridge vs RF × all features vs top 9) produced a clear result: **Ridge with all features dominates**. RF overfits at this sample size (train/test MAE ratio ~3×), and feature selection removes weak-but-real signal. The problem is well-suited to a linear model: the main drivers (`insulation_thickness`, `building_age`, `num_exterior_walls`, `insulation_grade`) have near-linear relationships with energy consumption, and the categorical gradients (construction period, insulation grade) are monotonic.

### Limitations

1. **Small sample (n = 1,200).** The train set has only 768 observations for 22 features. Cross-validation variance is non-negligible (CV MAE std = 1.0 kwh/sqm). More data would likely allow tree-based models to outperform Ridge.
2. **No temporal dimension.** We cannot test whether coefficients are stable across climate years or energy-price regimes.
3. **Winsorized target.** 2% of observations are capped at 310 kwh/sqm. The model cannot learn the true tail behaviour above this threshold.

### Next steps

1. **Quantile regression** for prediction intervals — the current ±19 kwh/sqm RMSE band is constant, but uncertainty likely varies with building type.
2. **Interaction terms** (`insulation_grade × building_age`, `construction_period × insulation_thickness`) could capture nonlinearities that Ridge misses without resorting to tree-based models.
3. **External validation** on a different city or climate zone to test geographic portability.